<a href="https://colab.research.google.com/github/shamilkaperera/Statistical-Learning-e23265/blob/main/E23265%20Assignment%20%237.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Analytical Derivation

### 1. Predicted State Distribution
$$m_k^- = \mathbb{E}[A_{k-1}x_{k-1}^+ + G_{k-1}w_{k-1}] = A_{k-1}\mathbb{E}[x_{k-1}^+] + G_{k-1}\mathbb{E}[w_{k-1}]$$
Since $\mathbb{E}[w_{k-1}] = 0$, $m_k^- = A_{k-1}m_{k-1}$.
$$P_k^- = \text{Var}(A_{k-1}x_{k-1}^+ + G_{k-1}w_{k-1}) = A_{k-1}\text{Var}(x_{k-1}^+)A_{k-1}^T + G_{k-1}\text{Var}(w_{k-1})G_{k-1}^T$$
$$P_k^- = A_{k-1}P_{k-1}A_{k-1}^T + G_{k-1}\Sigma_p G_{k-1}^T$$
Thus, $x_k^- \sim \mathscr{N}(m_k^-, P_k^-)$.

### 2. Predicted Measurement Distribution
$$\mathbb{E}[y_k^-] = \mathbb{E}[H_k x_k^- + z_k] = H_k m_k^-$$
$$\text{Var}(y_k^-) = H_k \text{Var}(x_k^-) H_k^T + \text{Var}(z_k) = H_k P_k^- H_k^T + \Sigma_m$$
Thus, $y_k^- \sim \mathscr{N}(H_k m_k^-, H_k P_k^- H_k^T + \Sigma_m)$.

### 3. Joint Distribution
$$\text{Cov}(x_k^-, y_k^-) = \text{Cov}(x_k^-, H_k x_k^- + z_k) = P_k^- H_k^T$$
$$\begin{bmatrix} x_k^- \\ y^{-}_k \end{bmatrix} \sim \mathscr{N}\left(
\begin{bmatrix} m_k^- \\ H_k m_k^- \end{bmatrix},
\begin{bmatrix} P_k^- & P_k^- H_k^T \\ H_k P_k^- & H_k P_k^- H_k^T + \Sigma_m \end{bmatrix}
\right)$$

### 4. & 5. Posterior State Distribution
Using multivariate Gaussian conditioning where $\Sigma_{AB}\Sigma_{BB}^{-1} = P_k^- H_k^T (H_k P_k^- H_k^T + \Sigma_m)^{-1} = K_k$:
* **Expected Value:** $\mathbb{E}[x^{-}_k \mid y^{-}_k = y^{\mathrm{obs}}_{k}] = m_k = m_k^- + K_k (y^{\mathrm{obs}}_{k} - H_k m_k^-)$
* **Variance:** $\text{Var}(x^{-}_k \mid y^{-}_k = y^{\mathrm{obs}}_{k}) = P_k = P_k^- - K_k H_k P_k^- = (I - K_k H_k) P_k^-$

##1-D Example

### 1. Prediction
$$m_k^- = a m_{k-1}$$
$$P_k^- = a P_{k-1} a^T + (1) q (1)^T = a^2 P_{k-1} + q$$

### 2. Update
With $S_k = h^2 P_k^- + r$ and $K_k = \frac{P_k^- h}{S_k}$:
$$m_k = m_k^- + K_k v_k = m_k^- + \frac{P_k^- h}{S_k}(y_k^{\mathrm{obs}} - h m_k^-)$$
$$P_k = P_k^- - K_k h P_k^- = (1 - K_k h) P_k^- = \left(1 - \frac{P_k^- h^2}{S_k}\right) P_k^-$$

### 3. Predictive Measurement Distribution
Mean $= h m_k^-$, Variance $= h^2 P_k^- + r$.
$$p(y^-_k | Y_{k-1})=\mathscr{N}(h m_k^-, h^2 P_k^- + r)$$

### 4. Posterior-predictive Measurement Distribution
Mean $= h m_k$, Variance $= h^2 P_k + r$.
$$p(y^-_k\mid Y_k)=\mathscr{N}(h m_k, h^2 P_k + r)$$

##1-D Example Code

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats as stats
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

a, q, h, r = 1.0, 0.5, 1.0, 1.0
x_true, m_k_minus_1, P_k_minus_1 = 0.0, 0.0, 2.0

fig, ax = plt.subplots(figsize=(8, 5))
x_axis = np.linspace(-10, 10, 500)
line_prior, = ax.plot([], [], label='Prior', color='blue')
line_post, = ax.plot([], [], label='Posterior', color='red')
line_meas = ax.axvline(x=0, color='green', linestyle='--', label='Measurement')

ax.set_xlim(-10, 10)
ax.set_ylim(0, 1.0)
ax.legend()

def update(frame):
    global x_true, m_k_minus_1, P_k_minus_1

    x_true = a * x_true + np.random.normal(0, np.sqrt(q))
    y_obs = h * x_true + np.random.normal(0, np.sqrt(r))

    m_k_minus = a * m_k_minus_1
    P_k_minus = (a**2) * P_k_minus_1 + q

    S_k = (h**2) * P_k_minus + r
    K_k = (P_k_minus * h) / S_k
    m_k = m_k_minus + K_k * (y_obs - h * m_k_minus)
    P_k = (1 - K_k * h) * P_k_minus

    line_prior.set_data(x_axis, stats.norm.pdf(x_axis, m_k_minus, np.sqrt(P_k_minus)))
    line_post.set_data(x_axis, stats.norm.pdf(x_axis, m_k, np.sqrt(P_k)))
    line_meas.set_xdata([y_obs, y_obs])

    m_k_minus_1, P_k_minus_1 = m_k, P_k
    return line_prior, line_post, line_meas

anim = FuncAnimation(fig, update, frames=20, interval=500, blit=True)
plt.close()
HTML(anim.to_jshtml())

##2D-Position Estimation

### State Transition Matrix (A):
Using $p(k) = p(k-1) + v(k-1)\Delta t + \frac{1}{2}a(k-1)\Delta t^2$ and $v(k) = v(k-1) + a(k-1)\Delta t$:
$$A = \begin{bmatrix}
1 & 0 & \Delta t & 0 \\
0 & 1 & 0 & \Delta t \\
0 & 0 & 1 & 0 \\
0 & 0 & 0 & 1
\end{bmatrix}$$

### Measurement Matrix (H):
Measuring only position ($p_x, p_y$):
$$H = \begin{bmatrix}
1 & 0 & 0 & 0 \\
0 & 1 & 0 & 0
\end{bmatrix}$$

### Noise Matrix (G):
Acceleration affects position by $\frac{1}{2}\Delta t^2$ and velocity by $\Delta t$:
$$G = \begin{bmatrix}
\frac{1}{2}\Delta t^2 & 0 \\
0 & \frac{1}{2}\Delta t^2 \\
\Delta t & 0 \\
0 & \Delta t
\end{bmatrix}$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def kalman_filter_2d(measurements, dt, q, r):
    A = np.array([[1, 0, dt, 0], [0, 1, 0, dt], [0, 0, 1, 0], [0, 0, 0, 1]])
    H = np.array([[1, 0, 0, 0], [0, 1, 0, 0]])
    G = np.array([[0.5 * dt**2, 0], [0, 0.5 * dt**2], [dt, 0], [0, dt]])

    Q = G @ (np.eye(2) * q) @ G.T
    R = np.eye(2) * r

    m = np.array([measurements[0, 0], measurements[0, 1], 0, 0])
    P = np.eye(4) * 10
    filtered_states = []

    for y_obs in measurements:
        m_minus = A @ m
        P_minus = A @ P @ A.T + Q

        S = H @ P_minus @ H.T + R
        K = P_minus @ H.T @ np.linalg.inv(S)

        m = m_minus + K @ (y_obs - H @ m_minus)
        P = (np.eye(4) - K @ H) @ P_minus
        filtered_states.append(m)

    return np.array(filtered_states)

# Generate synthetic GPS data
dt = 1.0
t = np.arange(0, 100, dt)
true_x, true_y = 100 * np.cos(t / 10), 100 * np.sin(t / 10)

measurements = np.vstack((
    true_x + np.random.normal(0, 5, len(t)),
    true_y + np.random.normal(0, 5, len(t))
)).T

filtered_output = kalman_filter_2d(measurements, dt, q=0.1, r=25.0)

# Plotting
plt.figure(figsize=(10, 8))
plt.plot(true_x, true_y, label="True Path", color='black', linestyle='dashed')
plt.scatter(measurements[:,0], measurements[:,1], label="Noisy GPS", color='red', alpha=0.5, s=15)
plt.plot(filtered_output[:, 0], filtered_output[:, 1], label="Filtered", color='blue', linewidth=2)
plt.legend()
plt.grid(True)
plt.show()